In [1]:
import getpass
import json
import time
from pathlib import Path

import pandas as pd
import requests

In [2]:
COMPANIES = {
    "Ford": {
        "ticker": "F",
        "cik": "0000037996"
    },
    "General Motors": {
        "ticker": "GM",
        "cik": "0001467858"
    },
    "Tesla": {
        "ticker": "TSLA",
        "cik": "0001318605"
    }
}

In [3]:
def find_repository_root(start_path=Path.cwd()):
    for folder in [start_path, *start_path.parents]:
        if (folder / ".git").exists():
            return folder
    
    raise FileNotFoundError(
        "Could not locate the Git repository."
    )


REPO_ROOT = find_repository_root()

XBRL_DATA_DIR = (
    REPO_ROOT /
    "data" /
    "raw" /
    "xbrl"
)

XBRL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("XBRL directory:", XBRL_DATA_DIR)

XBRL directory: D:\analytics\A_Python_Code\creditlens-rag\data\raw\xbrl


In [4]:
contact_email = getpass.getpass(
    "Enter your email for the SEC User-Agent: "
)

SEC_HEADERS = {
    "User-Agent": (
        f"CreditLens-RAG learning-project "
        f"{contact_email}"
    ),
    "Accept-Encoding": "gzip, deflate"
}

Enter your email for the SEC User-Agent:  ········


In [5]:
def download_company_facts(
    company_name,
    ticker,
    cik
):
    url = (
        "https://data.sec.gov/api/xbrl/"
        f"companyfacts/CIK{cik}.json"
    )
    
    response = requests.get(
        url,
        headers=SEC_HEADERS,
        timeout=60
    )
    
    response.raise_for_status()
    company_facts = response.json()
    
    output_path = (
        XBRL_DATA_DIR /
        f"{ticker}_companyfacts.json"
    )
    
    with output_path.open(
        "w",
        encoding="utf-8"
    ) as output_file:
        json.dump(
            company_facts,
            output_file
        )
    
    print(
        f"Downloaded {company_name}: "
        f"{output_path.name}"
    )
    
    return company_facts

In [6]:
company_facts_data = {}

for company_name, details in COMPANIES.items():
    
    company_facts_data[company_name] = (
        download_company_facts(
            company_name=company_name,
            ticker=details["ticker"],
            cik=details["cik"]
        )
    )
    
    time.sleep(0.5)

Downloaded Ford: F_companyfacts.json
Downloaded General Motors: GM_companyfacts.json
Downloaded Tesla: TSLA_companyfacts.json


In [7]:
ford_facts = company_facts_data["Ford"]

print("Entity name:", ford_facts["entityName"])
print(
    "Taxonomies:",
    ford_facts["facts"].keys()
)

Entity name: Ford Motor Co
Taxonomies: dict_keys(['dei', 'invest', 'us-gaap'])


In [8]:
ford_us_gaap = ford_facts["facts"]["us-gaap"]

print(
    "Ford US-GAAP concepts:",
    len(ford_us_gaap)
)

Ford US-GAAP concepts: 589


In [9]:
CANDIDATE_CONCEPTS = [
    "Assets",
    "Liabilities",
    "StockholdersEquity",
    "AssetsCurrent",
    "LiabilitiesCurrent",
    "CashAndCashEquivalentsAtCarryingValue",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Revenues",
    "NetIncomeLoss",
    "LongTermDebtCurrent",
    "LongTermDebtNoncurrent"
]

In [10]:
availability_records = []

for company_name, company_data in company_facts_data.items():
    
    available_concepts = set(
        company_data["facts"]["us-gaap"].keys()
    )
    
    for concept in CANDIDATE_CONCEPTS:
        availability_records.append({
            "company": company_name,
            "concept": concept,
            "available": (
                concept in available_concepts
            )
        })

availability_df = pd.DataFrame(
    availability_records
)

In [11]:
concept_availability = (
    availability_df
    .pivot(
        index="concept",
        columns="company",
        values="available"
    )
)

concept_availability

company,Ford,General Motors,Tesla
concept,,,
Assets,True,True,True
AssetsCurrent,True,True,True
CashAndCashEquivalentsAtCarryingValue,True,True,True
Liabilities,True,True,True
LiabilitiesCurrent,True,True,True
LongTermDebtCurrent,False,False,True
LongTermDebtNoncurrent,True,False,True
NetIncomeLoss,True,True,True
RevenueFromContractWithCustomerExcludingAssessedTax,True,True,True


In [13]:
def describe_concept(
    company_name,
    concept_name
):
    concept = (
        company_facts_data[company_name]
        ["facts"]
        ["us-gaap"]
        .get(concept_name)
    )
    
    if concept is None:
        return {
            "company": company_name,
            "concept": concept_name,
            "available": False
        }
    
    return {
        "company": company_name,
        "concept": concept_name,
        "available": True,
        "label": concept.get("label"),
        "description": concept.get("description"),
        "units": list(
            concept.get("units", {}).keys()
        )
    }

In [14]:
describe_concept(
    "Ford",
    "Assets"
)

{'company': 'Ford',
 'concept': 'Assets',
 'available': True,
 'label': 'Assets',
 'description': 'Sum of the carrying amounts as of the balance sheet date of all assets that are recognized. Assets are probable future economic benefits obtained or controlled by an entity as a result of past transactions or events.',
 'units': ['USD']}

In [15]:
FINANCIAL_CONCEPTS = {
    "assets": {
        "concept": "Assets",
        "period_type": "instant"
    },
    "current_assets": {
        "concept": "AssetsCurrent",
        "period_type": "instant"
    },
    "cash": {
        "concept": (
            "CashAndCashEquivalentsAtCarryingValue"
        ),
        "period_type": "instant"
    },
    "liabilities": {
        "concept": "Liabilities",
        "period_type": "instant"
    },
    "current_liabilities": {
        "concept": "LiabilitiesCurrent",
        "period_type": "instant"
    },
    "equity": {
        "concept": "StockholdersEquity",
        "period_type": "instant"
    },
    "revenue": {
        "concept": (
            "RevenueFromContractWithCustomer"
            "ExcludingAssessedTax"
        ),
        "period_type": "duration"
    },
    "net_income": {
        "concept": "NetIncomeLoss",
        "period_type": "duration"
    }
}

In [16]:
def extract_annual_fact(
    company_data,
    concept_name,
    year,
    period_type,
    unit="USD"
):
    concept_data = (
        company_data["facts"]
        ["us-gaap"]
        .get(concept_name)
    )
    
    if concept_data is None:
        return None
    
    unit_facts = (
        concept_data
        .get("units", {})
        .get(unit, [])
    )
    
    candidates = []
    
    for fact in unit_facts:
        
        if fact.get("form") != "10-K":
            continue
        
        if str(fact.get("fy")) != str(year):
            continue
        
        if fact.get("fp") != "FY":
            continue
        
        if not str(
            fact.get("end", "")
        ).startswith(str(year)):
            continue
        
        candidate = fact.copy()
        
        if period_type == "instant":
            if "start" in candidate:
                continue
        
        elif period_type == "duration":
            if "start" not in candidate:
                continue
            
            start_date = pd.Timestamp(
                candidate["start"]
            )
            end_date = pd.Timestamp(
                candidate["end"]
            )
            
            duration_days = (
                end_date - start_date
            ).days
            
            # Retain annual rather than quarterly facts
            if not 300 <= duration_days <= 400:
                continue
            
            candidate[
                "_duration_days"
            ] = duration_days
        
        candidates.append(candidate)
    
    if not candidates:
        return None
    
    candidates.sort(
        key=lambda fact: (
            fact.get("filed", ""),
            fact.get("_duration_days", 0)
        ),
        reverse=True
    )
    
    return candidates[0]

In [18]:
financial_fact_records = []

for company_name, details in COMPANIES.items():
    
    company_data = company_facts_data[
        company_name
    ]
    
    for year in [2023, 2024, 2025]:
        
        for metric_name, metric_info in (
            FINANCIAL_CONCEPTS.items()
        ):
            fact = extract_annual_fact(
                company_data=company_data,
                concept_name=metric_info["concept"],
                year=year,
                period_type=metric_info["period_type"]
            )
            
            financial_fact_records.append({
                "company": company_name,
                "ticker": details["ticker"],
                "year": year,
                "metric": metric_name,
                "concept": metric_info["concept"],
                "value": (
                    fact.get("val")
                    if fact is not None
                    else None
                ),
                "start_date": (
                    fact.get("start")
                    if fact is not None
                    else None
                ),
                "end_date": (
                    fact.get("end")
                    if fact is not None
                    else None
                ),
                "filed_date": (
                    fact.get("filed")
                    if fact is not None
                    else None
                ),
                "accession_number": (
                    fact.get("accn")
                    if fact is not None
                    else None
                )
            })

In [19]:
financial_facts_df = pd.DataFrame(
    financial_fact_records
)

financial_facts_df.head(10)

,company,ticker,year,metric,concept,value,start_date,end_date,filed_date,accession_number
0,Ford,F,2023,assets,Assets,2.733100e+11,None,2023-12-31,2024-02-07,0000037996-24-000009
1,Ford,F,2023,current_assets,AssetsCurrent,1.214810e+11,None,2023-12-31,2024-02-07,0000037996-24-000009
2,Ford,F,2023,cash,CashAndCashEquivalentsAtCarryingValue,2.486200e+10,None,2023-12-31,2024-02-07,0000037996-24-000009
3,Ford,F,2023,liabilities,Liabilities,2.305120e+11,None,2023-12-31,2024-02-07,0000037996-24-000009
4,Ford,F,2023,current_liabilities,LiabilitiesCurrent,1.015310e+11,None,2023-12-31,2024-02-07,0000037996-24-000009
5,Ford,F,2023,equity,StockholdersEquity,4.277300e+10,None,2023-12-31,2024-02-07,0000037996-24-000009
6,Ford,F,2023,revenue,RevenueFromContractWithCustomerExcludingAssess...,1.761910e+11,2023-01-01,2023-12-31,2024-02-07,0000037996-24-000009
7,Ford,F,2023,net_income,NetIncomeLoss,4.347000e+09,2023-01-01,2023-12-31,2024-02-07,0000037996-24-000009
8,Ford,F,2024,assets,Assets,2.851960e+11,None,2024-12-31,2025-02-06,0000037996-25-000013
9,Ford,F,2024,current_assets,AssetsCurrent,1.244740e+11,None,2024-12-31,2025-02-06,0000037996-25-000013


In [20]:
missing_facts = financial_facts_df[
    financial_facts_df["value"].isna()
]

print("Missing facts:", len(missing_facts))

missing_facts[
    [
        "company",
        "year",
        "metric",
        "concept"
    ]
]

Missing facts: 1


,company,year,metric,concept
23,Ford,2025,net_income,NetIncomeLoss


In [24]:
ford_us_gaap = (
    company_facts_data["Ford"]
    ["facts"]
    ["us-gaap"]
)

net_income_concepts = []

for concept_name, concept_data in ford_us_gaap.items():
    
    label = concept_data.get("label") or ""
    description = (
        concept_data.get("description") or ""
    )
    
    searchable_text = " ".join([
        concept_name,
        label,
        description
    ]).lower()
    
    if (
        "net income" in searchable_text
        or "profit loss" in searchable_text
    ):
        net_income_concepts.append({
            "concept": concept_name,
            "label": label
        })

pd.DataFrame(net_income_concepts)

,concept,label
0,AccumulatedOtherComprehensiveIncomeLossNetOfTax,"Accumulated Other Comprehensive Income (Loss),..."
1,AmortizationOfIntangibleAssets,Amortization of Intangible Assets
2,ComprehensiveIncomeNetOfTax,"Comprehensive Income (Loss), Net of Tax, Attri..."
3,ComprehensiveIncomeNetOfTaxAttributableToNonco...,"Comprehensive Income (Loss), Net of Tax, Attri..."
4,ComprehensiveIncomeNetOfTaxIncludingPortionAtt...,"Comprehensive Income (Loss), Net of Tax, Inclu..."
5,EarningsPerShareBasic,"Earnings Per Share, Basic"
6,EarningsPerShareDiluted,"Earnings Per Share, Diluted"
7,EquityMethodInvestmentSummarizedFinancialInfor...,"Equity Method Investment, Summarized Financial..."
8,IncomeLossFromContinuingOperationsPerBasicShare,"Income (Loss) from Continuing Operations, Per ..."
9,IncomeLossFromContinuingOperationsPerDilutedShare,"Income (Loss) from Continuing Operations, Per ..."


In [25]:
FORD_NET_INCOME_CANDIDATES = [
    "NetIncomeLoss",
    "ProfitLoss",
    "NetIncomeLossAvailableToCommonStockholdersBasic"
]

candidate_results = []

for concept_name in FORD_NET_INCOME_CANDIDATES:
    
    fact = extract_annual_fact(
        company_data=company_facts_data["Ford"],
        concept_name=concept_name,
        year=2025,
        period_type="duration"
    )
    
    candidate_results.append({
        "concept": concept_name,
        "value": (
            fact.get("val")
            if fact is not None
            else None
        ),
        "start": (
            fact.get("start")
            if fact is not None
            else None
        ),
        "end": (
            fact.get("end")
            if fact is not None
            else None
        ),
        "filed": (
            fact.get("filed")
            if fact is not None
            else None
        ),
        "accession": (
            fact.get("accn")
            if fact is not None
            else None
        )
    })

pd.DataFrame(candidate_results)

,concept,value,start,end,filed,accession
0,NetIncomeLoss,NaN,None,None,None,None
1,ProfitLoss,-8.162000e+09,2025-01-01,2025-12-31,2026-02-11,0000037996-26-000015
2,NetIncomeLossAvailableToCommonStockholdersBasic,-8.182000e+09,2025-01-01,2025-12-31,2026-02-11,0000037996-26-000015


In [26]:
FINANCIAL_CONCEPTS[
    "net_income"
]["concepts"] = [
    "NetIncomeLoss",
    "ProfitLoss",
    "NetIncomeLossAvailableToCommonStockholdersBasic"
]

In [21]:
financial_values_df = (
    financial_facts_df
    .pivot(
        index=[
            "company",
            "ticker",
            "year"
        ],
        columns="metric",
        values="value"
    )
    .reset_index()
)

financial_values_df.columns.name = None

financial_values_df = (
    financial_values_df
    .sort_values(
        ["company", "year"]
    )
    .reset_index(drop=True)
)

financial_values_df

,company,ticker,year,assets,cash,current_assets,current_liabilities,equity,liabilities,net_income,revenue
0,Ford,F,2023,2.733100e+11,2.486200e+10,1.214810e+11,1.015310e+11,4.277300e+10,2.305120e+11,4.347000e+09,1.761910e+11
1,Ford,F,2024,2.851960e+11,2.293500e+10,1.244740e+11,1.068590e+11,4.483500e+10,2.403380e+11,5.879000e+09,1.849920e+11
2,Ford,F,2025,2.891600e+11,2.335600e+10,1.234870e+11,1.148900e+11,3.595200e+10,2.531800e+11,NaN,1.872670e+11
3,General Motors,GM,2023,2.730640e+11,1.885300e+10,1.016180e+11,9.444500e+10,6.428600e+10,2.047570e+11,1.012700e+10,1.576580e+11
4,General Motors,GM,2024,2.797610e+11,1.987200e+10,1.085450e+11,9.626500e+10,6.307200e+10,2.141710e+11,6.008000e+09,1.716060e+11
5,General Motors,GM,2025,2.812840e+11,2.094500e+10,1.087670e+11,9.334200e+10,6.111900e+10,2.181160e+11,2.697000e+09,1.679710e+11
6,Tesla,TSLA,2023,1.066180e+11,1.639800e+10,4.961600e+10,2.874800e+10,6.263400e+10,4.300900e+10,1.499700e+10,9.677300e+10
7,Tesla,TSLA,2024,1.220700e+11,1.613900e+10,5.836000e+10,2.882100e+10,7.291300e+10,4.839000e+10,7.091000e+09,9.769000e+10
8,Tesla,TSLA,2025,1.378060e+11,1.651300e+10,6.864200e+10,3.171400e+10,8.213700e+10,5.494100e+10,3.794000e+09,9.482700e+10


In [31]:
def extract_metric_fact(
    company_data,
    metric_info,
    year
):
    concept_candidates = metric_info.get(
        "concepts",
        [metric_info["concept"]]
    )
    
    for concept_name in concept_candidates:
        
        fact = extract_annual_fact(
            company_data=company_data,
            concept_name=concept_name,
            year=year,
            period_type=metric_info[
                "period_type"
            ]
        )
        
        if fact is not None:
            return fact, concept_name
    
    return None, None

In [32]:
financial_fact_records = []

for company_name, details in COMPANIES.items():
    
    company_data = company_facts_data[
        company_name
    ]
    
    for year in [2023, 2024, 2025]:
        
        for metric_name, metric_info in (
            FINANCIAL_CONCEPTS.items()
        ):
            fact, concept_used = (
                extract_metric_fact(
                    company_data=company_data,
                    metric_info=metric_info,
                    year=year
                )
            )
            
            financial_fact_records.append({
                "company": company_name,
                "ticker": details["ticker"],
                "year": year,
                "metric": metric_name,
                "concept_used": concept_used,
                "value": (
                    fact.get("val")
                    if fact is not None
                    else None
                ),
                "start_date": (
                    fact.get("start")
                    if fact is not None
                    else None
                ),
                "end_date": (
                    fact.get("end")
                    if fact is not None
                    else None
                ),
                "filed_date": (
                    fact.get("filed")
                    if fact is not None
                    else None
                ),
                "accession_number": (
                    fact.get("accn")
                    if fact is not None
                    else None
                )
            })
            

In [29]:
financial_facts_df = pd.DataFrame(
    financial_fact_records
)

missing_facts = financial_facts_df[
    financial_facts_df["value"].isna()
]

print("Missing facts:", len(missing_facts))

missing_facts

Missing facts: 0


,company,ticker,year,metric,concept_used,value,start_date,end_date,filed_date,accession_number


In [30]:
financial_facts_df[
    (financial_facts_df["company"] == "Ford")
    &
    (financial_facts_df["year"] == 2025)
    &
    (financial_facts_df["metric"] == "net_income")
]

,company,ticker,year,metric,concept_used,value,start_date,end_date,filed_date,accession_number
23,Ford,F,2025,net_income,ProfitLoss,-8162000000,2025-01-01,2025-12-31,2026-02-11,0000037996-26-000015


In [33]:
financial_values_df = (
    financial_facts_df
    .pivot(
        index=[
            "company",
            "ticker",
            "year"
        ],
        columns="metric",
        values="value"
    )
    .reset_index()
)

financial_values_df.columns.name = None

financial_values_df = (
    financial_values_df
    .sort_values(
        ["company", "year"]
    )
    .reset_index(drop=True)
)

financial_values_df

,company,ticker,year,assets,cash,current_assets,current_liabilities,equity,liabilities,net_income,revenue
0,Ford,F,2023,273310000000,24862000000,121481000000,101531000000,42773000000,230512000000,4347000000,176191000000
1,Ford,F,2024,285196000000,22935000000,124474000000,106859000000,44835000000,240338000000,5879000000,184992000000
2,Ford,F,2025,289160000000,23356000000,123487000000,114890000000,35952000000,253180000000,-8162000000,187267000000
3,General Motors,GM,2023,273064000000,18853000000,101618000000,94445000000,64286000000,204757000000,10127000000,157658000000
4,General Motors,GM,2024,279761000000,19872000000,108545000000,96265000000,63072000000,214171000000,6008000000,171606000000
5,General Motors,GM,2025,281284000000,20945000000,108767000000,93342000000,61119000000,218116000000,2697000000,167971000000
6,Tesla,TSLA,2023,106618000000,16398000000,49616000000,28748000000,62634000000,43009000000,14997000000,96773000000
7,Tesla,TSLA,2024,122070000000,16139000000,58360000000,28821000000,72913000000,48390000000,7091000000,97690000000
8,Tesla,TSLA,2025,137806000000,16513000000,68642000000,31714000000,82137000000,54941000000,3794000000,94827000000


In [34]:
financial_values_df["current_ratio"] = (
    financial_values_df["current_assets"]
    /
    financial_values_df["current_liabilities"]
)

financial_values_df[
    "liabilities_to_equity"
] = (
    financial_values_df["liabilities"]
    /
    financial_values_df["equity"]
)

financial_values_df["cash_to_assets"] = (
    financial_values_df["cash"]
    /
    financial_values_df["assets"]
)

financial_values_df["net_margin"] = (
    financial_values_df["net_income"]
    /
    financial_values_df["revenue"]
)

financial_values_df["revenue_growth"] = (
    financial_values_df
    .groupby("company")["revenue"]
    .pct_change()
)

In [35]:
ratio_results = financial_values_df[
    [
        "company",
        "year",
        "current_ratio",
        "liabilities_to_equity",
        "cash_to_assets",
        "net_margin",
        "revenue_growth"
    ]
].copy()

ratio_results[
    [
        "cash_to_assets",
        "net_margin",
        "revenue_growth"
    ]
] = (
    ratio_results[
        [
            "cash_to_assets",
            "net_margin",
            "revenue_growth"
        ]
    ] * 100
)

ratio_results = ratio_results.round(2)

ratio_results

,company,year,current_ratio,liabilities_to_equity,cash_to_assets,net_margin,revenue_growth
0,Ford,2023,1.20,5.39,9.10,2.47,NaN
1,Ford,2024,1.16,5.36,8.04,3.18,5.00
2,Ford,2025,1.07,7.04,8.08,-4.36,1.23
3,General Motors,2023,1.08,3.19,6.90,6.42,NaN
4,General Motors,2024,1.13,3.40,7.10,3.50,8.85
5,General Motors,2025,1.17,3.57,7.45,1.61,-2.12
6,Tesla,2023,1.73,0.69,15.38,15.50,NaN
7,Tesla,2024,2.02,0.66,13.22,7.26,0.95
8,Tesla,2025,2.16,0.67,11.98,4.00,-2.93


In [36]:
facts_output_path = (
    REPO_ROOT /
    "data" /
    "financial_facts.csv"
)

metrics_output_path = (
    REPO_ROOT /
    "data" /
    "financial_metrics.csv"
)

financial_facts_df.to_csv(
    facts_output_path,
    index=False
)

financial_values_df.to_csv(
    metrics_output_path,
    index=False
)

print("Saved:", facts_output_path)
print("Saved:", metrics_output_path)

Saved: D:\analytics\A_Python_Code\creditlens-rag\data\financial_facts.csv
Saved: D:\analytics\A_Python_Code\creditlens-rag\data\financial_metrics.csv
